# Simulated helmet shop

## Reviews

In [ ]:
import numpy as np

RNG = np.random.default_rng(1)
STARS = np.arange(1, 6)
BASE = np.array([0.09, 0.04, 0.07, 0.18, 0.62])   # share of 1★..5★ for an average helmet
CUTS = np.log(np.cumsum(BASE)[:-1] / (1 - np.cumsum(BASE)[:-1]))


def star_probs(q):
    """P(1★..5★) for a helmet of hidden quality q (q = 0 is an average helmet)."""
    q = np.atleast_1d(q)[:, None]
    cdf = 1 / (1 + np.exp(-(CUTS[None, :] - q)))
    cdf = np.hstack([np.zeros((len(q), 1)), cdf, np.ones((len(q), 1))])
    return np.diff(cdf, axis=1)


def mean_of(q):
    return (star_probs(q) * STARS).sum(axis=1)


def q_for_mean(target):
    """The hidden quality whose expected rating is `target`."""
    lo, hi = -6.0, 6.0
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if mean_of(mid)[0] < target else (lo, mid)
    return (lo + hi) / 2


def reviews(q, n):
    return RNG.choice(STARS, size=n, p=star_probs(q)[0])

## Shop and helmets A, B

In [ ]:
N_HELMETS = 2000
Q = RNG.normal(0.0, 0.7, N_HELMETS)
COUNTS = np.clip(np.round(RNG.lognormal(np.log(25), 1.5, N_HELMETS)), 1, 5000).astype(int)
SHOP = [reviews(q, n) for q, n in zip(Q, COUNTS)]
AVG = np.array([r.mean() for r in SHOP])

A = np.array([5] * 10)
Q_B = q_for_mean(4.6)
B = reviews(Q_B, 1800)
_diff = int(round(4.6 * len(B))) - int(B.sum())          # nudge a few 4s/5s so B is exactly 4.60
_idx = np.flatnonzero(B == (4 if _diff > 0 else 5))[:abs(_diff)]
B[_idx] += 1 if _diff > 0 else -1
print(f"Helmet A: {A.mean():.1f} from {len(A)} reviews.  Helmet B: {B.mean():.2f} from {len(B):,} reviews.")

## Luck test

In [ ]:
LUCK = np.array([reviews(Q_B, 10).mean() for _ in range(1000)])
print(f"{int((LUCK == 5.0).sum())} of 1,000 identical helmets came out at a perfect 5.0, just by luck")

ORDER = np.argsort(-AVG, kind="stable")
print(f"{int((COUNTS[ORDER[:20]] <= 5).sum())} of the shop's top 20 (sorted by rating) have 5 reviews or fewer")

## Bayesian average: `(C·m + sum of ratings) / (C + n)`

In [ ]:
ALL = np.concatenate(SHOP)
M = ALL.mean()
SIGMA2 = np.mean([r.var(ddof=1) for r in SHOP if len(r) > 1])      # one review's noise
TAU2 = max(AVG.var() - np.mean(SIGMA2 / COUNTS), 1e-6)              # true spread between helmets
C = SIGMA2 / TAU2


def score(r, c=None, m=None):
    c, m, r = C if c is None else c, M if m is None else m, np.asarray(r)
    return (c * m + r.sum()) / (c + len(r))


print(f"m = {M:.2f}  (the typical score in this shop)")
print(f"one review is off by about ±{np.sqrt(SIGMA2):.2f} stars;  real helmets differ by about ±{np.sqrt(TAU2):.2f}")
print(f"C = ({np.sqrt(SIGMA2):.2f} / {np.sqrt(TAU2):.2f})² = {C:.1f} pretend reviews")
print(f"a new helmet with 1, 10, 100 five-star reviews scores {score([5]):.2f}, {score([5] * 10):.2f}, {score([5] * 100):.2f}")

## Fives A needs to beat B

In [ ]:
def fives_to_beat_B(c=None, m=None):
    c, m = C if c is None else c, M if m is None else m
    b = score(B, c, m)
    return max(0, int(np.floor(c * (b - m) / (5 - b))) + 1)


print(f"Helmet A: 5.0 → {score(A):.2f}.   Helmet B: {B.mean():.2f} → {score(B):.2f}")
print(f"A needs {fives_to_beat_B()} five-star reviews to beat B; it has {len(A)}")

_a = RNG.normal(score(A), np.sqrt(SIGMA2) / np.sqrt(C + len(A)), 200_000)
_b = RNG.normal(score(B), np.sqrt(SIGMA2) / np.sqrt(C + len(B)), 200_000)
print(f"chance A is really the better helmet: {np.mean(np.clip(_a, 1, 5) > np.clip(_b, 1, 5)):.0%}")

## Other C and m

In [ ]:
for c, m in [(C, M), (2 * C, M), (1.0, M), (C, 3.5), (C, 4.5)]:
    top = np.argsort(-np.array([score(r, c, m) for r in SHOP]), kind="stable")[:10]
    print(f"C = {c:5.1f}, m = {m:.1f}:  A needs {fives_to_beat_B(c, m):3d} fives to beat B;  "
          f"the top ten have a median of {int(np.median(COUNTS[top]))} reviews")

In [ ]:
MY_C, MY_M = 5, 4.1
print(f"with C = {MY_C} and m = {MY_M}, helmet A needs {fives_to_beat_B(MY_C, MY_M)} five-star reviews to beat B")